# Role Gap Analysis

Compare a target job's salary and scope against comparable roles within the same company.

**Usage:** Set `JOB_ID` and `CSV_PATH` in the next cell, then Run All.

In [ ]:
# ── CONFIG ──────────────────────────────────────────────────────────
JOB_ID = "2689707b-7314-4246-ac95-1e6466970ba3"
CSV_PATH = "crusoe_salaries.csv"
# ────────────────────────────────────────────────────────────────────

import re
from collections import Counter

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
import seaborn as sns

from classify import (
    add_classifications, add_usd_salary, TO_USD, SENIORITY_ORDER,
    normalize_department,
)

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (14, 6)
plt.rcParams["figure.dpi"] = 120

# Load data
df = pd.read_csv(CSV_PATH)
df["job_id"] = df["job_id"].astype(str)

# Ensure classification columns exist
if "department" not in df.columns:
    df = add_classifications(df)
if "department_raw" not in df.columns:
    df["department_raw"] = df["department"]
if "min_usd" not in df.columns:
    df = add_usd_salary(df)

# Ensure we have salary data
df = df.dropna(subset=["salary_min", "salary_max"])
df["mid_usd"] = (df["salary_min"] + df["salary_max"]) / 2

# Validate target exists
assert JOB_ID in df["job_id"].values, f"JOB_ID {JOB_ID!r} not found in {CSV_PATH}"
target = df[df["job_id"] == JOB_ID].iloc[0]

print(f"Target: {target['title']}")
print(f"Department: {target['department']} ({target.get('department_raw', '')})")
print(f"Salary: ${target['salary_min']:,.0f} – ${target['salary_max']:,.0f} (mid ${target['mid_usd']:,.0f})")
print(f"Location: {target['location']}")
print(f"Loaded {len(df)} roles with salary from {CSV_PATH}")

## 1. Scope Scoring

Each role is scored 0–10 on **ownership and scope** based on description language.

In [ ]:
# ── Scope scoring ───────────────────────────────────────────────────
BUILDER_PATTERNS = [
    r"from scratch", r"ground up", r"greenfield", r"first hire",
    r"founding", r"build out", r"build\b.*from",
]
OWNER_PATTERNS = [
    r"\bown\b", r"define the strategy", r"set the vision",
    r"\broadmap\b", r"lead the\b", r"shape the",
]
LEADER_PATTERNS = [
    r"hire and manage", r"build a team", r"cross-functional leadership",
    r"manage a team", r"grow the team",
]
CONTRIBUTOR_PATTERNS = [
    r"contribute to", r"\bassist\b", r"join a team",
    r"report to", r"work under", r"support the team",
]


def score_scope(desc: str) -> int:
    if not isinstance(desc, str):
        return 5  # neutral default
    desc_lower = desc.lower()
    score = 5  # start at midpoint
    for p in BUILDER_PATTERNS:
        if re.search(p, desc_lower):
            score += 2
    for p in OWNER_PATTERNS:
        if re.search(p, desc_lower):
            score += 2
    for p in LEADER_PATTERNS:
        if re.search(p, desc_lower):
            score += 1
    for p in CONTRIBUTOR_PATTERNS:
        if re.search(p, desc_lower):
            score -= 1
    return max(0, min(10, score))


df["scope_score"] = df["description_md"].apply(score_scope)
target_scope = df.loc[df["job_id"] == JOB_ID, "scope_score"].iloc[0]

print(f"Target scope score: {target_scope}/10")
print(f"\nScope distribution:")
print(df["scope_score"].describe().round(1))

## 2. Comparable Roles

Roles in the same department with similar scope (±2). Relaxes if too few matches.

In [ ]:
# ── Find comparables ────────────────────────────────────────────────
target_dept = target["department"]
SCOPE_TOLERANCE = 2

# Primary: same department + similar scope
comparables = df[
    (df["department"] == target_dept)
    & (df["scope_score"] >= target_scope - SCOPE_TOLERANCE)
    & (df["scope_score"] <= target_scope + SCOPE_TOLERANCE)
    & (df["job_id"] != JOB_ID)
].copy()

# Relax if too few
if len(comparables) < 5:
    SCOPE_TOLERANCE = 3
    comparables = df[
        (df["department"] == target_dept)
        & (df["scope_score"] >= target_scope - SCOPE_TOLERANCE)
        & (df["scope_score"] <= target_scope + SCOPE_TOLERANCE)
        & (df["job_id"] != JOB_ID)
    ].copy()

if len(comparables) < 5:
    # Drop department filter, keep scope
    comparables = df[
        (df["scope_score"] >= target_scope - SCOPE_TOLERANCE)
        & (df["scope_score"] <= target_scope + SCOPE_TOLERANCE)
        & (df["job_id"] != JOB_ID)
    ].copy()
    print(f"⚠ Relaxed to cross-department (scope ±{SCOPE_TOLERANCE}): {len(comparables)} comparables")
else:
    print(f"Found {len(comparables)} comparables in {target_dept} (scope ±{SCOPE_TOLERANCE})")

# Add target back for plotting
plot_df = pd.concat([comparables, df[df["job_id"] == JOB_ID]], ignore_index=True)
plot_df = plot_df.sort_values("mid_usd", ascending=True)
print(f"\nComparable salary range: ${comparables['mid_usd'].min():,.0f} – ${comparables['mid_usd'].max():,.0f}")
print(f"Target midpoint: ${target['mid_usd']:,.0f}")

## 3. Salary Comparison

In [ ]:
# ── Salary range bars ───────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, max(4, len(plot_df) * 0.55)))

colors = ["#d62728" if str(row["job_id"]) == JOB_ID else "#4a7cc9"
          for _, row in plot_df.iterrows()]

for i, (_, row) in enumerate(plot_df.iterrows()):
    ax.barh(i, row["salary_max"] - row["salary_min"], left=row["salary_min"],
            color=colors[i], alpha=0.85, height=0.6, edgecolor="white")
    ax.text(row["salary_max"] + 2000, i,
            f"${row['mid_usd']:,.0f}",
            va="center", fontsize=9, color="#333")

ax.set_yticks(range(len(plot_df)))
ax.set_yticklabels([f"{row['title'][:50]}" for _, row in plot_df.iterrows()], fontsize=9)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x/1000:.0f}K"))
ax.set_xlabel("Salary Range (USD)")
ax.set_title(f"Salary Ranges: {target['title']} vs. Comparables", fontsize=13, fontweight="bold")

# Legend
legend_patches = [
    mpatches.Patch(color="#d62728", label=f"Target: {target['title'][:40]}"),
    mpatches.Patch(color="#4a7cc9", label="Comparables"),
]
ax.legend(handles=legend_patches, loc="lower right", fontsize=9)
plt.tight_layout()
plt.show()

## 4. Company-Wide Salary Percentile

In [ ]:
# ── Percentile placement ────────────────────────────────────────────
percentile = (df["mid_usd"] <= target["mid_usd"]).mean() * 100

fig, ax = plt.subplots(figsize=(14, 5))
sns.histplot(df["mid_usd"], bins=35, kde=True, ax=ax, color="#4a7cc9", alpha=0.6)

ax.axvspan(target["salary_min"], target["salary_max"], color="#d62728", alpha=0.25,
           label=f"Target range")
ax.axvline(target["mid_usd"], color="#d62728", linewidth=2, linestyle="--",
           label=f"Target mid: ${target['mid_usd']:,.0f} (P{percentile:.0f})")

ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x/1000:.0f}K"))
ax.set_xlabel("Salary Midpoint (USD)")
ax.set_ylabel("Number of Roles")
ax.set_title(f"Where {target['title'][:40]} Sits in the Company Pay Landscape",
             fontsize=13, fontweight="bold")
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

print(f"Target is at the {percentile:.0f}th percentile of all {len(df)} salaried roles.")

## 5. Experience vs. Salary

In [ ]:
# ── Experience vs salary bubble chart ───────────────────────────────
def extract_yoe(desc):
    if not isinstance(desc, str):
        return np.nan
    m = re.search(r"(\d+)\+?\s*years?", desc)
    return int(m.group(1)) if m else np.nan

plot_df["yoe"] = plot_df["description_md"].apply(extract_yoe)
plot_df["range_usd"] = plot_df["salary_max"] - plot_df["salary_min"]
bubble_df = plot_df.dropna(subset=["yoe"])

if len(bubble_df) >= 3:
    fig, ax = plt.subplots(figsize=(12, 8))

    is_target = bubble_df["job_id"].astype(str) == JOB_ID
    sizes = bubble_df["range_usd"] / 500

    ax.scatter(bubble_df.loc[~is_target, "yoe"], bubble_df.loc[~is_target, "mid_usd"],
               s=sizes[~is_target], alpha=0.6, color="#4a7cc9", edgecolors="white", linewidth=0.5)
    ax.scatter(bubble_df.loc[is_target, "yoe"], bubble_df.loc[is_target, "mid_usd"],
               s=sizes[is_target], alpha=0.9, color="#d62728", edgecolors="black", linewidth=1.5,
               zorder=5, label="Target")

    for _, row in bubble_df.iterrows():
        ax.annotate(row["title"][:30], (row["yoe"], row["mid_usd"]),
                    fontsize=7, alpha=0.7, ha="center", va="bottom",
                    xytext=(0, 8), textcoords="offset points")

    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x/1000:.0f}K"))
    ax.set_xlabel("Years of Experience Required")
    ax.set_ylabel("Salary Midpoint (USD)")
    ax.set_title("Experience vs. Salary (bubble size = salary range width)",
                 fontsize=13, fontweight="bold")
    ax.legend(fontsize=10)
    plt.tight_layout()
    plt.show()
else:
    print("Too few roles with YoE data for bubble chart.")

## 6. Skills Overlap

Auto-detects keywords from the target role's description and scores all comparables.

In [ ]:
# ── Auto-detected skills heatmap ────────────────────────────────────
# Extract 2-3 word phrases from the target description as skill signals
def extract_skill_phrases(desc, top_n=12):
    """Extract frequent meaningful 2-3 word phrases from a description."""
    if not isinstance(desc, str):
        return {}
    # Normalize
    text = desc.lower()
    text = re.sub(r"[^a-z\s-]", " ", text)

    # Stop words to skip
    stops = {"the", "and", "for", "with", "that", "this", "you", "will",
             "our", "are", "from", "have", "your", "about", "been", "more",
             "their", "into", "also", "who", "can", "all", "has", "than",
             "its", "may", "other", "new", "not", "but", "what", "which",
             "when", "how", "each", "some", "such", "both", "between",
             "should", "would", "could", "across", "within", "including",
             "through", "well", "these", "those", "ability", "experience",
             "work", "working", "team", "role", "join", "looking", "ideal",
             "candidate", "strong", "years", "minimum", "preferred", "plus"}

    words = [w for w in text.split() if w not in stops and len(w) > 2]

    # Bigrams
    bigrams = [f"{words[i]} {words[i+1]}" for i in range(len(words)-1)]
    counts = Counter(bigrams)

    # Filter to meaningful phrases (appear 2+ times)
    phrases = {p: c for p, c in counts.items() if c >= 2}
    return dict(sorted(phrases.items(), key=lambda x: -x[1])[:top_n])


target_desc = target.get("description_md", "")
skill_phrases = extract_skill_phrases(target_desc)

if skill_phrases and len(comparables) > 0:
    # Score each comparable on these phrases
    roles_to_score = pd.concat([df[df["job_id"] == JOB_ID], comparables]).drop_duplicates(subset="job_id")
    roles_to_score = roles_to_score.head(20)  # cap for readability

    scores = {}
    for _, row in roles_to_score.iterrows():
        desc = str(row.get("description_md", "")).lower()
        role_scores = {}
        for phrase, _ in skill_phrases.items():
            role_scores[phrase] = len(re.findall(re.escape(phrase), desc))
        scores[row["title"][:45]] = role_scores

    skills_df = pd.DataFrame(scores).T
    skills_df = skills_df.loc[:, skills_df.sum() > 0]  # drop zero columns

    if not skills_df.empty:
        fig, ax = plt.subplots(figsize=(14, max(4, len(skills_df) * 0.5)))
        sns.heatmap(skills_df, annot=True, fmt="d", cmap="YlOrRd",
                    linewidths=1, ax=ax, cbar_kws={"label": "Keyword Count"})
        ax.set_title(f"Skills Overlap: {target['title'][:40]} vs. Comparables",
                     fontsize=13, fontweight="bold")
        ax.set_ylabel("")
        plt.tight_layout()
        plt.show()
    else:
        print("No overlapping skill phrases found.")
else:
    print("Insufficient data for skills heatmap.")

## 7. Scope vs. Salary

All roles plotted by ownership scope and salary. Comparables are highlighted.

In [ ]:
# ── Scope vs salary scatter ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 8))

# All roles in grey
other = df[~df["job_id"].isin(comparables["job_id"].tolist() + [JOB_ID])]
ax.scatter(other["scope_score"], other["mid_usd"],
           s=30, alpha=0.15, color="grey", label="Other roles")

# Comparables in blue
ax.scatter(comparables["scope_score"], comparables["mid_usd"],
           s=60, alpha=0.6, color="#4a7cc9", edgecolors="white",
           linewidth=0.5, label="Comparables")

# Target in red
ax.scatter(target_scope, target["mid_usd"],
           s=150, color="#d62728", edgecolors="black", linewidth=1.5,
           zorder=5, label=f"Target: {target['title'][:35]}")

# Label comparables
for _, row in comparables.iterrows():
    ax.annotate(row["title"][:25], (row["scope_score"], row["mid_usd"]),
                fontsize=7, alpha=0.7, ha="center", va="bottom",
                xytext=(0, 8), textcoords="offset points")

ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x/1000:.0f}K"))
ax.set_xlabel("Scope Score (0 = contributor, 10 = builder/owner)", fontsize=11)
ax.set_ylabel("Salary Midpoint (USD)", fontsize=11)
ax.set_title("Scope vs. Salary: Where Does This Role Sit?",
             fontsize=13, fontweight="bold")
ax.legend(fontsize=10, loc="upper left")
ax.set_xlim(-0.5, 10.5)
plt.tight_layout()
plt.show()

# Summary stats
comp_mid = comparables["mid_usd"].median()
gap = target["mid_usd"] - comp_mid
pct = (gap / comp_mid * 100) if comp_mid else 0
direction = "above" if gap > 0 else "below"
print(f"Target midpoint ${target['mid_usd']:,.0f} is ${abs(gap):,.0f} ({abs(pct):.1f}%) {direction} comparable median ${comp_mid:,.0f}")